---
## Perturbation Illustrations

5 rows (perturbation type) × 3 columns (clean | low severity | high severity) using a
representative Freeman Center frame.

Reads pre-rendered PNGs from `results/figures/frames/freeman/` — run `render_figures.ipynb`
first if those files are missing.

In [ ]:
import matplotlib.image as mpimg

plt.rcParams.update({"font.family": "serif", "font.size": 9})

FRAMES_BASE = REPO_ROOT / "results" / "figures" / "frames" / "freeman"
FRAME_IDX   = "frame_001.png"

PERTURBATION_TYPES = [
    ("brightness",     "Brightness\nReduction"),
    ("fog",            "Fog"),
    ("gaussian_noise", "Gaussian\nNoise"),
    ("motion_blur",    "Motion\nBlur"),
    ("rain",           "Rain"),
]
COL_HEADERS = ["Clean", "Low Severity", "High Severity"]
N_ROWS, N_COLS = len(PERTURBATION_TYPES), 3

CELL_W, CELL_H, LABEL_W = 3.8, 2.15, 1.0
fig_w = LABEL_W + N_COLS * CELL_W
fig_h = N_ROWS * CELL_H + 0.45

fig, axes = plt.subplots(
    N_ROWS, N_COLS, figsize=(fig_w, fig_h),
    gridspec_kw={"left": LABEL_W / fig_w, "right": 0.995,
                 "top": 0.94, "bottom": 0.01, "hspace": 0.06, "wspace": 0.03},
)

clean_img = mpimg.imread(str(FRAMES_BASE / "detection" / FRAME_IDX))

for row, (ptype, plabel) in enumerate(PERTURBATION_TYPES):
    images = [
        clean_img,
        mpimg.imread(str(FRAMES_BASE / "perturbations" / f"{ptype}_low"  / FRAME_IDX)),
        mpimg.imread(str(FRAMES_BASE / "perturbations" / f"{ptype}_high" / FRAME_IDX)),
    ]
    for col, img in enumerate(images):
        ax = axes[row][col]
        ax.imshow(img, aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row][0].set_ylabel(plabel, fontsize=9, fontfamily="serif",
                            rotation=90, va="center", ha="center", labelpad=6)

for col, header in enumerate(COL_HEADERS):
    axes[0][col].set_title(
        header, fontsize=9.5, fontfamily="serif", pad=4,
        fontweight="bold" if col == 0 else "normal",
        color="#444" if col == 0 else "#1565C0" if col == 1 else "#B71C1C",
    )

fig.savefig(PLATES_DIR / "perturbation_illustrations.pdf", dpi=300, bbox_inches="tight")
fig.savefig(PLATES_DIR / "perturbation_illustrations.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved → {PLATES_DIR}/perturbation_illustrations.{{pdf,png}}")

---
## Tracking Identity Strip

Four consecutive frames from CVB clip `0749` (frames 31–34) showing OC-SORT identity
continuity through partial occlusion. Frames 31–33 carry 3 active tracks; frame 34
shows 8 tracks after re-identification.

Reads from `data/processed/tracking_v2/cvb/` and `data/raw/cvb/raw_frames/`.

In [ ]:
import json
import cv2
from tracking.visualize_tracks import render_frame

plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.titlesize": 9})

VIDEO_ID   = "0749_arm01_gopro1_20200325_025919_beh3_ani5_ins3_cut_10"
FRAME_IDS  = [31, 32, 33, 34]
TRACK_JSON = REPO_ROOT / "data" / "processed" / "tracking_v2" / "cvb" / f"{VIDEO_ID}_tracks.json"
FRAMES_DIR = REPO_ROOT / "data" / "raw" / "cvb" / "raw_frames" / VIDEO_ID
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)"]

with open(TRACK_JSON) as f:
    track_data = json.load(f)
frames_dict = track_data["frames"]

rendered = []
for fid in FRAME_IDS:
    img_path = FRAMES_DIR / f"img_{fid:05d}.jpg"
    if not img_path.exists():
        raise FileNotFoundError(f"Frame not found: {img_path}")
    bgr = cv2.imread(str(img_path))
    dets = frames_dict.get(str(fid), [])
    annotated_rgb = cv2.cvtColor(render_frame(bgr, dets), cv2.COLOR_BGR2RGB)
    rendered.append((fid, annotated_rgb, len(dets)))

fig, axes = plt.subplots(1, 4, figsize=(14.0, 3.5))
for ax, (fid, rgb, n_tracks), label in zip(axes, rendered, PANEL_LABELS):
    ax.imshow(rgb)
    ax.text(0.02, 0.97, f"Frame {fid}", transform=ax.transAxes,
            fontsize=7, fontfamily="serif", color="white", verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.55, linewidth=0))
    ax.set_xlabel(label, fontsize=10, fontfamily="serif", labelpad=4)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

fig.tight_layout(pad=0.3, w_pad=0.3)
fig.set_size_inches(14.0, 3.5)
fig.savefig(PLATES_DIR / "tracking_identity_strip.pdf", dpi=300)
fig.savefig(PLATES_DIR / "tracking_identity_strip.png", dpi=300)
plt.show()
print(f"Saved → {PLATES_DIR}/tracking_identity_strip.{{pdf,png}}")
print(f"\nVideo: {VIDEO_ID}")
for fid, _, n in rendered:
    print(f"  Frame {fid}: {n} active tracks")

In [1]:
import os, json, glob, shutil
import numpy as np
import cv2

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
DATA_RAW = os.path.join(REPO_ROOT, "data", "raw")
TRACKING_V2 = os.path.join(REPO_ROOT, "data", "processed", "tracking_v2")
FRAMES_DIR = os.path.join(REPO_ROOT, "results", "figures", "frames")
STAGING_DIR = os.path.join(FRAMES_DIR, "freeman_staging")
MANIFEST_PATH = os.path.join(FRAMES_DIR, "manifest.json")

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(STAGING_DIR, exist_ok=True)

N_SELECT = 4  # frames to include per dataset in manifest
N_EXTRACT = 4  # frames to extract per Freeman AVI

print("REPO_ROOT:", REPO_ROOT)
print("FRAMES_DIR:", FRAMES_DIR)
print("STAGING_DIR:", STAGING_DIR)

REPO_ROOT: /home/sakif/TXST/Thesis/cattle-vision-framework
FRAMES_DIR: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames
STAGING_DIR: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/freeman_staging


## Freeman Center — Frame Extraction from AVI files

Extract `N_EXTRACT` evenly-spaced frames from each AVI in `data/raw/freeman-cmb-2024/freeman-raw-videos/`
and write them to `results/figures/frames/freeman_staging/`.


In [2]:
FREEMAN_RAW = os.path.join(DATA_RAW, "freeman-cmb-2024", "freeman-raw-videos")
avi_files = sorted(glob.glob(os.path.join(FREEMAN_RAW, "*.avi")))
print(f"Found {len(avi_files)} AVI files in {FREEMAN_RAW}")

extracted = []
for avi_path in avi_files:
    stem = os.path.splitext(os.path.basename(avi_path))[0]
    cap = cv2.VideoCapture(avi_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < 2:
        cap.release()
        continue
    indices = np.linspace(0, total_frames - 1, N_EXTRACT, dtype=int)
    for idx in indices:
        out_name = f"{stem}_frame{idx:06d}.jpg"
        out_path = os.path.join(STAGING_DIR, out_name)
        if not os.path.exists(out_path):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                cv2.imwrite(out_path, frame)
        if os.path.exists(out_path):
            extracted.append(out_path)
    cap.release()

print(
    f"Extracted {len(extracted)} frames from {len(avi_files)} AVI files → {STAGING_DIR}"
)

Found 54 AVI files in /home/sakif/TXST/Thesis/cattle-vision-framework/data/raw/freeman-cmb-2024/freeman-raw-videos
Extracted 216 frames from 54 AVI files → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/freeman_staging


## Frame Selector

For **CBVD-5** and **CVB**: scan tracking JSONs in `data/processed/tracking_v2/` and select frames
where ≥ 2 cattle are visible, evenly spaced across selected videos.  
For **OOD datasets** (OpenCows2020, Cows2021, CattleEyeView, Freeman): sample evenly from image pool.


In [3]:
import pandas as pd


def even_sample(items, n):
    """Return n evenly-spaced items from a sorted list."""
    items = sorted(items)
    if len(items) <= n:
        return items
    indices = np.linspace(0, len(items) - 1, n, dtype=int)
    return [items[i] for i in indices]


# Pre-load val-split video IDs from behavior prediction CSVs so frame selection
# is restricted to videos that actually have behavior predictions (Pass D).
_PRED_RUNS = {
    "cbvd5": os.path.join(
        REPO_ROOT, "results", "behavior", "predictions", "videomae_cbvd5_v2_val.csv"
    ),
    "cvb": os.path.join(
        REPO_ROOT, "results", "behavior", "predictions", "videomae_cvb_v2_val.csv"
    ),
}
_val_video_ids = {}
for ds, path in _PRED_RUNS.items():
    df = pd.read_csv(path)
    _val_video_ids[ds] = set(df["video_id"].astype(str).unique())
    print(f"{ds}: {len(_val_video_ids[ds])} val-split videos in predictions CSV")


def select_from_tracking(dataset_key, n=N_SELECT):
    """Return n frame entries using tracking JSONs, preferring frames with >=2 cattle.
    Restricted to videos present in the behavior predictions CSV (val split only)."""
    tracking_dir = os.path.join(TRACKING_V2, dataset_key)
    json_files = sorted(glob.glob(os.path.join(tracking_dir, "*_tracks.json")))
    allowed_vids = _val_video_ids.get(dataset_key)  # None for OOD datasets

    candidates = []  # (image_path, track_count, video_id, frame_key)
    for jpath in json_files:
        with open(jpath) as f:
            data = json.load(f)
        vid = data["video_id"]
        # Skip videos not in the val-split predictions
        if allowed_vids is not None and str(vid) not in allowed_vids:
            continue
        for fkey, dets in data["frames"].items():
            n_cattle = len(dets)
            if n_cattle < 1:
                continue
            # Resolve image path
            if dataset_key == "cbvd5":
                img_path = os.path.join(
                    DATA_RAW,
                    "cbvd5",
                    "labelframes",
                    "labelframes",
                    f"{vid}_{int(fkey):05d}.jpg",
                )
            elif dataset_key == "cvb":
                img_path = os.path.join(
                    DATA_RAW, "cvb", "raw_frames", vid, f"img_{int(fkey):05d}.jpg"
                )
            else:
                continue
            if os.path.exists(img_path):
                candidates.append((img_path, n_cattle, vid, fkey))

    # Prefer frames with >=2 cattle, then fall back to any
    multi = [c for c in candidates if c[1] >= 2]
    pool = multi if multi else candidates
    chosen = even_sample(pool, n)

    return [
        {
            "path": p,
            "has_tracking": True,
            "n_cattle": nc,
            "video_id": vid,
            "frame": fk,
        }
        for p, nc, vid, fk in chosen
    ]


def select_from_images(image_pool, n=N_SELECT, dataset_key=""):
    """Return n evenly-spaced entries from a flat image pool."""
    chosen = even_sample(image_pool, n)
    return [
        {
            "path": p,
            "has_tracking": False,
            "n_cattle": None,
        }
        for p in chosen
    ]


# ── CBVD-5 ────────────────────────────────────────────────────────────────────
cbvd5_entries = select_from_tracking("cbvd5")
print(f"CBVD-5:     {len(cbvd5_entries)} frames selected")

# ── CVB ───────────────────────────────────────────────────────────────────────
cvb_entries = select_from_tracking("cvb")
print(f"CVB:        {len(cvb_entries)} frames selected")

# ── OpenCows2020 ──────────────────────────────────────────────────────────────
opencow_imgs = sorted(
    glob.glob(
        os.path.join(
            DATA_RAW, "opencow2020", "detection_and_localisation", "img", "*.jpg"
        )
    )
)
opencow_entries = select_from_images(opencow_imgs, dataset_key="opencows2020")
print(
    f"OpenCows2020: {len(opencow_entries)} frames selected (pool={len(opencow_imgs)})"
)

# ── Cows2021 ──────────────────────────────────────────────────────────────────
cows2021_imgs = sorted(
    glob.glob(
        os.path.join(
            DATA_RAW, "cows2021", "detection_and_localisation-val", "img", "*.jpg"
        )
    )
)
cows2021_entries = select_from_images(cows2021_imgs, dataset_key="cows2021")
print(
    f"Cows2021:   {len(cows2021_entries)} frames selected (pool={len(cows2021_imgs)})"
)

# ── CattleEyeView ─────────────────────────────────────────────────────────────
cev_imgs = sorted(
    glob.glob(
        os.path.join(
            DATA_RAW,
            "cattle-eye-view",
            "dataset",
            "detect",
            "images",
            "test",
            "*",
            "*.jpg",
        )
    )
)
cev_entries = select_from_images(cev_imgs, dataset_key="cattleeyeview")
print(f"CattleEyeView: {len(cev_entries)} frames selected (pool={len(cev_imgs)})")

# ── Freeman Center ─────────────────────────────────────────────────────────────
freeman_imgs = sorted(glob.glob(os.path.join(STAGING_DIR, "*.jpg")))
freeman_entries = select_from_images(freeman_imgs, dataset_key="freeman")
print(f"Freeman:    {len(freeman_entries)} frames selected (pool={len(freeman_imgs)})")

# ── mmCows ────────────────────────────────────────────────────────────────────
mmcows_imgs = sorted(
    glob.glob(os.path.join(DATA_RAW, "mmCows", "images", "0725", "cam_4", "*.jpg"))
)
mmcows_entries = select_from_images(mmcows_imgs, dataset_key="mmcows")
print(f"mmCows:     {len(mmcows_entries)} frames selected (pool={len(mmcows_imgs)})")

cbvd5: 50 val-split videos in predictions CSV
cvb: 88 val-split videos in predictions CSV


CBVD-5:     4 frames selected


CVB:        4 frames selected
OpenCows2020: 4 frames selected (pool=7043)
Cows2021:   4 frames selected (pool=1023)
CattleEyeView: 4 frames selected (pool=2490)
Freeman:    4 frames selected (pool=216)
mmCows:     4 frames selected (pool=5040)


## Smoke Test

Print per-dataset frame counts; raise an error if any dataset has 0 frames.


In [4]:
SELECTION = {
    "cbvd5": cbvd5_entries,
    "cvb": cvb_entries,
    "opencows2020": opencow_entries,
    "cows2021": cows2021_entries,
    "cattleeyeview": cev_entries,
    "freeman": freeman_entries,
    "mmcows": mmcows_entries,
}

print("Per-dataset frame counts:")
total = 0
failures = []
for ds, entries in SELECTION.items():
    n = len(entries)
    total += n
    status = "✓" if n >= 3 else ("⚠ WARN (<3)" if n > 0 else "✗ FAIL (0)")
    print(f"  {ds:16s}: {n:3d} frames  {status}")
    if n == 0:
        failures.append(ds)

print(f"\nTotal: {total} frames across {len(SELECTION)} datasets")
if failures:
    raise RuntimeError(
        f"Zero frames selected for datasets: {failures}. Check data paths."
    )
else:
    print("Smoke test PASSED — all datasets have ≥ 1 frame.")

Per-dataset frame counts:
  cbvd5           :   4 frames  ✓
  cvb             :   4 frames  ✓
  opencows2020    :   4 frames  ✓
  cows2021        :   4 frames  ✓
  cattleeyeview   :   4 frames  ✓
  freeman         :   4 frames  ✓
  mmcows          :   4 frames  ✓

Total: 28 frames across 7 datasets
Smoke test PASSED — all datasets have ≥ 1 frame.


## Manifest Writer

Write `results/figures/frames/manifest.json` with the structure:

```json
{"cbvd5": {"frame_001": {"path": "...", "has_tracking": true, ...}, ...}, ...}
```


In [5]:
manifest = {}
for ds, entries in SELECTION.items():
    manifest[ds] = {}
    for i, entry in enumerate(entries, start=1):
        key = f"frame_{i:03d}"
        manifest[ds][key] = entry

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

size_kb = os.path.getsize(MANIFEST_PATH) / 1024
print(f"Manifest written to: {MANIFEST_PATH}  ({size_kb:.1f} KB)")
print(f"Datasets in manifest: {list(manifest.keys())}")
for ds, frames in manifest.items():
    sample = list(frames.values())[0]
    print(f'  {ds}: {len(frames)} frames, sample path: ...{sample["path"][-60:]}')

Manifest written to: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/manifest.json  (6.4 KB)
Datasets in manifest: ['cbvd5', 'cvb', 'opencows2020', 'cows2021', 'cattleeyeview', 'freeman', 'mmcows']
  cbvd5: 4 frames, sample path: ...amework/data/raw/cbvd5/labelframes/labelframes/341_00002.jpg
  cvb: 4 frames, sample path: ...01_gopro1_20200322_222554_beh7_ani1_ins1_cut_1/img_00001.jpg
  opencows2020: 4 frames, sample path: ...ta/raw/opencow2020/detection_and_localisation/img/000001.jpg
  cows2021: 4 frames, sample path: ...ta/raw/cows2021/detection_and_localisation-val/img/07248.jpg
  cattleeyeview: 4 frames, sample path: .../cattle-eye-view/dataset/detect/images/test/01.mp4/00014.jpg
  freeman: 4 frames, sample path: ...res/frames/freeman_staging/D0C1.02(Gr,G,DHF)_frame000000.jpg
  mmcows: 4 frames, sample path: ...rk/data/raw/mmCows/images/0725/cam_4/1690271846_02-57-26.jpg


---

## Overlay Render Passes

Five passes build progressively on each frame in the manifest:

| Pass | Output dir                                   | Content                                                          |
| ---- | -------------------------------------------- | ---------------------------------------------------------------- |
| A    | `{dataset}/detection/`                       | RF-DETR-Seg boxes + confidence scores; masks cached              |
| B    | `{dataset}/det_seg/`                         | RF-DETR-Seg masks + boxes (uniform across all 6 datasets)        |
| C    | `{dataset}/det_seg_track/`                   | RF-DETR-Seg masks + track IDs (CBVD-5/CVB) or `N/A` (OOD)        |
| D    | `{dataset}/full_pipeline/`                   | RF-DETR-Seg masks + track IDs + behavior label (CBVD-5/CVB only) |
| E    | `{dataset}/perturbations/{type}_{severity}/` | Perturbed frame + detection boxes + delta_mAP50 annotation       |

Each pass checks for existing output before writing (checkpoint behaviour).  
The smoke-test cell below verifies the manifest, then defines shared helpers and caches.


In [6]:
import sys, json, os
import numpy as np
import cv2

sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

# Re-load manifest (idempotent)
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
assert manifest, f"Manifest empty — run manifest writer first: {MANIFEST_PATH}"

print(f"Manifest: {MANIFEST_PATH}")
for ds, frames in manifest.items():
    print(f"  {ds:16s}: {len(frames)} frames")
print(
    f"\nTotal: {sum(len(v) for v in manifest.values())} frames across {len(manifest)} datasets"
)

# Shared constants ────────────────────────────────────────────────────────────
TRACKING_DATASETS = {"cbvd5", "cvb"}
CONF_THRESH = 0.3
SEG_CKPT = os.path.join(
    REPO_ROOT, "runs", "seg_medium_lr5e5", "checkpoint_best_ema.pth"
)

# Max long-edge before resizing for RF-DETR-Seg inference.
# Masks are generated at input resolution; 4480×2800 images OOM on 12 GB GPU.
MAX_INFER_SIZE = 1920

LABEL_NAMES = {
    0: "Standing",
    1: "Lying",
    2: "Foraging",
    3: "Drinking",
    4: "Ruminating",
    5: "Grooming",
    6: "Other",
}

PALETTE_BGR = [
    (50, 220, 0),
    (255, 80, 0),
    (0, 220, 255),
    (200, 0, 255),
    (0, 160, 255),
    (0, 200, 200),
    (255, 180, 0),
    (60, 0, 255),
]


def cap_pil_for_inference(pil_img, max_size=MAX_INFER_SIZE):
    """Resize PIL image so its longest edge ≤ max_size, preserving aspect ratio.
    Returns the (possibly resized) image and the scale factor applied."""
    w, h = pil_img.size
    long_edge = max(w, h)
    if long_edge <= max_size:
        return pil_img, 1.0
    scale = max_size / long_edge
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    from PIL import Image as _PIL
    return pil_img.resize((new_w, new_h), _PIL.LANCZOS), scale


# Drawing helpers ─────────────────────────────────────────────────────────────
def draw_boxes(img_bgr, boxes_xyxy, scores, extra_labels=None):
    """Return copy of img_bgr with RF-DETR boxes and confidence/label text."""
    out = img_bgr.copy()
    for idx, (box, score) in enumerate(zip(boxes_xyxy, scores)):
        x1, y1, x2, y2 = (int(v) for v in box)
        color = PALETTE_BGR[idx % len(PALETTE_BGR)]
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        text = extra_labels[idx] if extra_labels else f"{score:.2f}"
        cv2.putText(
            out,
            text,
            (x1, max(y1 - 5, 14)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            1,
            cv2.LINE_AA,
        )
    return out


def overlay_masks(img_bgr, masks, alpha=0.4):
    """Return new image with semi-transparent colored masks overlaid."""
    out = img_bgr.copy()
    for idx, mask in enumerate(masks):
        if mask is None or not np.any(mask):
            continue
        color = PALETTE_BGR[idx % len(PALETTE_BGR)]
        layer = out.copy()
        layer[mask.astype(bool)] = color
        out = cv2.addWeighted(layer, alpha, out, 1 - alpha, 0)
    return out


def add_behavior_banner(img_bgr, behavior_label):
    """Add a dark banner at the top of the image with the behavior label."""
    out = img_bgr.copy()
    h, w = out.shape[:2]
    bh = max(32, int(h * 0.055))
    cv2.rectangle(out, (0, 0), (w, bh), (30, 30, 30), -1)
    cv2.putText(
        out,
        f"Behavior: {behavior_label}",
        (10, bh - 9),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    return out


# In-memory caches shared across pass cells
det_cache = {}  # (ds, fkey) → {'boxes': [...], 'scores': [...]}
mask_cache = {}  # (ds, fkey) → [np.ndarray bool, ...]
track_cache = {}  # (ds, fkey) → [(bbox, track_id), ...]   CBVD-5/CVB only

print("\nHelpers and caches initialised. Ready for render passes.")
print(f"RF-DETR-Seg checkpoint: {SEG_CKPT}")
print(f"  exists: {os.path.exists(SEG_CKPT)}")
print(f"MAX_INFER_SIZE: {MAX_INFER_SIZE} px (long edge)")

Manifest: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/manifest.json
  cbvd5           : 4 frames
  cvb             : 4 frames
  opencows2020    : 4 frames
  cows2021        : 4 frames
  cattleeyeview   : 4 frames
  freeman         : 4 frames
  mmcows          : 4 frames

Total: 28 frames across 7 datasets

Helpers and caches initialised. Ready for render passes.
RF-DETR-Seg checkpoint: /home/sakif/TXST/Thesis/cattle-vision-framework/runs/seg_medium_lr5e5/checkpoint_best_ema.pth
  exists: True
MAX_INFER_SIZE: 1920 px (long edge)


## Pass A — Detection + Segmentation Inference (RF-DETR-Seg)

Load `runs/seg_medium_lr5e5/checkpoint_best_ema.pth` (Config B EMA, best mAP@50:95 85.0%).  
Run inference on all 24 selected frames. Cache boxes, scores, and masks for downstream passes.  
Save detection-only overlays (boxes + scores) to `{dataset}/detection/`.


In [7]:
import torch
from PIL import Image
from rfdetr import RFDETRSegMedium

print(f"Loading RF-DETR-Seg from {SEG_CKPT} ...")
# num_classes=1 (1 fg + 1 bg = 2 outputs matching the checkpoint's class_embed shape).
# pretrain_weights=None skips the base rf-detr-seg-medium.pt load, which would otherwise
# reinitialize the detection head to 91 COCO classes before we overwrite with our weights.
detector = RFDETRSegMedium(num_classes=1, pretrain_weights=None)
_ckpt = torch.load(SEG_CKPT, map_location="cpu", weights_only=False)
detector.model.model.load_state_dict(_ckpt["model"], strict=True)
detector.model.model.eval()
print(f"  loaded (epoch {_ckpt['epoch']}).\n")

for ds, frames in manifest.items():
    out_dir = os.path.join(FRAMES_DIR, ds, "detection")
    os.makedirs(out_dir, exist_ok=True)
    for fkey, entry in frames.items():
        img_path = entry["path"]
        if not os.path.exists(img_path):
            print(f"  [WARN] missing: {img_path}")
            det_cache[(ds, fkey)] = {"boxes": [], "scores": []}
            mask_cache[(ds, fkey)] = []
            continue

        pil_img = Image.open(img_path).convert("RGB")
        src_w, src_h = pil_img.size

        # Resize for inference if image exceeds MAX_INFER_SIZE (e.g. mmCows 4480×2800)
        infer_img, scale = cap_pil_for_inference(pil_img)
        dets = detector.predict(infer_img, threshold=CONF_THRESH)

        if dets and dets.xyxy is not None and len(dets.xyxy):
            # Scale boxes back to source resolution
            boxes = [
                [float(v) / scale for v in row] for row in dets.xyxy
            ]
            scores = [float(s) for s in dets.confidence]
            # Scale masks back to source resolution
            if dets.mask is not None and len(dets.mask):
                masks = []
                for m in dets.mask:
                    if scale < 1.0:
                        import cv2 as _cv2
                        m_up = _cv2.resize(
                            m.astype(np.uint8), (src_w, src_h),
                            interpolation=_cv2.INTER_NEAREST,
                        ).astype(bool)
                        masks.append(m_up)
                    else:
                        masks.append(m.astype(bool))
            else:
                masks = []
        else:
            boxes, scores, masks = [], [], []

        det_cache[(ds, fkey)] = {"boxes": boxes, "scores": scores}
        mask_cache[(ds, fkey)] = masks

        out_path = os.path.join(out_dir, f"{fkey}.png")
        if os.path.exists(out_path):
            print(f"  [skip] {ds}/{fkey}")
            continue

        img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        rendered = draw_boxes(img_bgr, boxes, scores)
        cv2.imwrite(out_path, rendered)
        print(f"  [A] {ds}/{fkey}: {len(boxes)} det, {len(masks)} masks → {out_path}")

print("\nPass A complete.")

[2026-05-31 05:36:18] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-31 05:36:18] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Loading RF-DETR-Seg from /home/sakif/TXST/Thesis/cattle-vision-framework/runs/seg_medium_lr5e5/checkpoint_best_ema.pth ...


[2026-05-31 05:36:19] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


  loaded (epoch 58).



...

  [skip] freeman/frame_002
  [skip] freeman/frame_003
  [skip] freeman/frame_004


  [A] mmcows/frame_001: 3 det, 3 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/detection/frame_001.png


  [A] mmcows/frame_002: 5 det, 5 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/detection/frame_002.png


  [A] mmcows/frame_003: 3 det, 3 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/detection/frame_003.png


  [A] mmcows/frame_004: 2 det, 2 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/detection/frame_004.png

Pass A complete.


## Pass B — Detection + Segmentation Overlay

Uses RF-DETR-Seg masks cached by Pass A — uniform across all 6 datasets, no SAM2 needed.  
For **CBVD-5 and CVB**: also loads tracking JSONs to populate `track_cache` (used by Passes C and D).  
Saves overlaid renders to `{dataset}/det_seg/`.


In [8]:
for ds, frames in manifest.items():
    out_dir = os.path.join(FRAMES_DIR, ds, "det_seg")
    os.makedirs(out_dir, exist_ok=True)

    for fkey, entry in frames.items():
        img_path = entry["path"]
        if not os.path.exists(img_path):
            print(f"  [WARN] missing: {img_path}")
            continue

        img_bgr = cv2.imread(img_path)
        masks = mask_cache.get((ds, fkey), [])
        cache = det_cache.get((ds, fkey), {"boxes": [], "scores": []})

        # For CBVD-5/CVB: load tracking JSON to populate track_cache for Passes C and D
        if ds in TRACKING_DATASETS and (ds, fkey) not in track_cache:
            vid = entry["video_id"]
            frame = entry["frame"]
            j_path = os.path.join(TRACKING_V2, ds, f"{vid}_tracks.json")
            with open(j_path) as f:
                trk = json.load(f)
            trk_dets = trk["frames"].get(str(frame), [])
            track_cache[(ds, fkey)] = [(d["bbox"], d["track_id"]) for d in trk_dets]

        out_path = os.path.join(out_dir, f"{fkey}.png")
        if os.path.exists(out_path):
            print(f"  [skip] {ds}/{fkey}")
            continue

        canvas = overlay_masks(img_bgr, masks)
        canvas = draw_boxes(canvas, cache["boxes"], cache["scores"])
        cv2.imwrite(out_path, canvas)
        print(f"  [B] {ds}/{fkey}: {len(masks)} masks → {out_path}")

print("\nPass B complete.")

  [skip] cbvd5/frame_001
  [skip] cbvd5/frame_002
  [skip] cbvd5/frame_003
  [skip] cbvd5/frame_004
  [skip] cvb/frame_001
  [skip] cvb/frame_002
  [skip] cvb/frame_003
  [skip] cvb/frame_004
  [skip] opencows2020/frame_001
  [skip] opencows2020/frame_002
  [skip] opencows2020/frame_003
  [skip] opencows2020/frame_004
  [skip] cows2021/frame_001
  [skip] cows2021/frame_002
  [skip] cows2021/frame_003
  [skip] cows2021/frame_004
  [skip] cattleeyeview/frame_001
  [skip] cattleeyeview/frame_002
  [skip] cattleeyeview/frame_003
  [skip] cattleeyeview/frame_004
  [skip] freeman/frame_001
  [skip] freeman/frame_002
  [skip] freeman/frame_003
  [skip] freeman/frame_004


  [B] mmcows/frame_001: 3 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg/frame_001.png


  [B] mmcows/frame_002: 5 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg/frame_002.png


  [B] mmcows/frame_003: 3 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg/frame_003.png


  [B] mmcows/frame_004: 2 masks → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg/frame_004.png

Pass B complete.


## Pass C — Detection + Segmentation + Tracking ID

For **CBVD-5 and CVB**: RF-DETR-Seg masks + tracking IDs from `tracking_v2` JSONs.  
For **OOD datasets**: RF-DETR-Seg masks + boxes + `"Tracking: N/A"` text.


In [9]:
for ds, frames in manifest.items():
    out_dir = os.path.join(FRAMES_DIR, ds, "det_seg_track")
    os.makedirs(out_dir, exist_ok=True)

    for fkey, entry in frames.items():
        img_path = entry["path"]
        if not os.path.exists(img_path):
            print(f"  [WARN] missing: {img_path}")
            continue

        img_bgr = cv2.imread(img_path)
        masks = mask_cache.get((ds, fkey), [])

        if ds in TRACKING_DATASETS:
            trk_entries = track_cache.get((ds, fkey), [])
            trk_boxes = [e[0] for e in trk_entries]
            trk_labels = [f"ID:{e[1]}" for e in trk_entries]
            canvas = overlay_masks(img_bgr, masks)
            canvas = draw_boxes(
                canvas, trk_boxes, [1.0] * len(trk_boxes), extra_labels=trk_labels
            )
        else:
            cache = det_cache.get((ds, fkey), {"boxes": [], "scores": []})
            canvas = overlay_masks(img_bgr, masks)
            canvas = draw_boxes(canvas, cache["boxes"], cache["scores"])
            h, w = canvas.shape[:2]
            cv2.putText(
                canvas,
                "Tracking: N/A",
                (10, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (120, 120, 120),
                1,
                cv2.LINE_AA,
            )

        out_path = os.path.join(out_dir, f"{fkey}.png")
        if os.path.exists(out_path):
            print(f"  [skip] {ds}/{fkey}")
            continue
        cv2.imwrite(out_path, canvas)
        print(f"  [C] {ds}/{fkey} → {out_path}")

print("\nPass C complete.")

  [skip] cbvd5/frame_001
  [skip] cbvd5/frame_002
  [skip] cbvd5/frame_003
  [skip] cbvd5/frame_004
  [skip] cvb/frame_001
  [skip] cvb/frame_002
  [skip] cvb/frame_003


  [skip] cvb/frame_004
  [skip] opencows2020/frame_001
  [skip] opencows2020/frame_002


  [skip] opencows2020/frame_003
  [skip] opencows2020/frame_004
  [skip] cows2021/frame_001
  [skip] cows2021/frame_002
  [skip] cows2021/frame_003
  [skip] cows2021/frame_004
  [skip] cattleeyeview/frame_001
  [skip] cattleeyeview/frame_002
  [skip] cattleeyeview/frame_003
  [skip] cattleeyeview/frame_004
  [skip] freeman/frame_001
  [skip] freeman/frame_002
  [skip] freeman/frame_003
  [skip] freeman/frame_004


  [C] mmcows/frame_001 → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg_track/frame_001.png


  [C] mmcows/frame_002 → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg_track/frame_002.png


  [C] mmcows/frame_003 → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg_track/frame_003.png


  [C] mmcows/frame_004 → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/det_seg_track/frame_004.png

Pass C complete.


## Pass D — Full Pipeline (CBVD-5 and CVB only)

Render original + RF-DETR-Seg masks + tracking IDs + behavior label from  
`results/behavior/predictions_sam2_tubelets/videomae_*_v2_val.csv`,  
matched by `video_id` + frame range. Falls back to `"Unknown"` if no match.


In [10]:
import pandas as pd

PRED_RUNS = {
    "cbvd5": os.path.join(
        REPO_ROOT, "results", "behavior", "predictions", "videomae_cbvd5_v2_val.csv"
    ),
    "cvb": os.path.join(
        REPO_ROOT, "results", "behavior", "predictions", "videomae_cvb_v2_val.csv"
    ),
}
pred_dfs = {ds: pd.read_csv(p) for ds, p in PRED_RUNS.items() if os.path.exists(p)}


def lookup_behavior(ds, video_id, frame_num):
    df = pred_dfs.get(ds)
    if df is None:
        return "Unknown"
    match = df[
        (df["video_id"].astype(str) == str(video_id))
        & (df["start_frame"] <= int(frame_num))
        & (df["end_frame"] >= int(frame_num))
    ]
    return (
        LABEL_NAMES.get(int(match.iloc[0]["pred_label_id"]), "Unknown")
        if not match.empty
        else "Unknown"
    )


for ds in ["cbvd5", "cvb"]:
    frames = manifest[ds]
    out_dir = os.path.join(FRAMES_DIR, ds, "full_pipeline")
    os.makedirs(out_dir, exist_ok=True)

    for fkey, entry in frames.items():
        img_path = entry["path"]
        if not os.path.exists(img_path):
            print(f"  [WARN] missing: {img_path}")
            continue

        img_bgr = cv2.imread(img_path)
        masks = mask_cache.get((ds, fkey), [])
        trk_data = track_cache.get((ds, fkey), [])
        trk_boxes = [e[0] for e in trk_data]
        trk_labels = [f"ID:{e[1]}" for e in trk_data]
        behavior = lookup_behavior(ds, entry["video_id"], entry["frame"])

        canvas = overlay_masks(img_bgr, masks)
        canvas = draw_boxes(
            canvas, trk_boxes, [1.0] * len(trk_boxes), extra_labels=trk_labels
        )
        canvas = add_behavior_banner(canvas, behavior)

        out_path = os.path.join(out_dir, f"{fkey}.png")
        if os.path.exists(out_path):
            print(f"  [skip] {ds}/{fkey}")
            continue
        cv2.imwrite(out_path, canvas)
        print(f"  [D] {ds}/{fkey}: {behavior} → {out_path}")

print("\nPass D complete.")

  [skip] cbvd5/frame_001
  [skip] cbvd5/frame_002
  [skip] cbvd5/frame_003
  [skip] cbvd5/frame_004
  [skip] cvb/frame_001


  [skip] cvb/frame_002
  [skip] cvb/frame_003
  [skip] cvb/frame_004

Pass D complete.


## Pass E — Perturbation Overlay Renders

For each selected frame × each dataset × 10 perturbation conditions (5 types × 2 severities):
apply the albumentations transform in-memory, run RF-DETR detection-only, and save to
`results/figures/frames/{dataset}/perturbations/{type}_{severity}/`.

Each rendered image is annotated with the perturbation type, severity, and `delta_mAP50`
from `results/generalization/perturbation_delta.csv`.

Perturbation parameters exactly match `scripts/eval_detection_perturbation.py`.
Already-rendered frames are skipped on re-run.


In [11]:
import pandas as pd
import albumentations as A
from PIL import Image as _PILImage

PERTURB_DELTA_CSV = os.path.join(
    REPO_ROOT, "results", "generalization", "perturbation_delta.csv"
)
_PERTURB_TYPES = ["brightness", "gaussian_noise", "motion_blur", "fog", "rain"]
_SEVERITIES = ["low", "high"]

# Load delta_mAP50 lookup: (dataset, perturbation_type, severity) → delta
_delta_df = pd.read_csv(PERTURB_DELTA_CSV)
_delta_lookup = {
    (row.dataset, row.perturbation_type, row.severity): row.delta_mAP50
    for _, row in _delta_df.iterrows()
}
print(f"Loaded {len(_delta_lookup)} delta entries from {PERTURB_DELTA_CSV}")


def _make_perturb_transform(perturbation, severity):
    """Return PIL Image → perturbed PIL Image. Parameters match eval_detection_perturbation.py."""
    if perturbation == "brightness":
        limit = -0.50 if severity == "low" else -0.75
        aug = A.RandomBrightnessContrast(
            brightness_limit=(limit, limit), contrast_limit=0, p=1.0
        )
    elif perturbation == "gaussian_noise":
        sigma = 25 if severity == "low" else 50
        std = sigma / 255.0
        aug = A.GaussNoise(std_range=(std, std), p=1.0)
    elif perturbation == "motion_blur":
        k = 7 if severity == "low" else 15
        aug = A.MotionBlur(blur_limit=(k, k), p=1.0)
    elif perturbation == "fog":
        coef = 0.3 if severity == "low" else 0.6
        aug = A.RandomFog(fog_coef_range=(coef, coef), p=1.0)
    elif perturbation == "rain":
        if severity == "low":
            aug = A.RandomRain(
                slant_range=(-10, 10), drop_length=20, drop_width=1, p=1.0
            )
        else:
            aug = A.RandomRain(
                slant_range=(-20, 20), drop_length=40, drop_width=2, p=1.0
            )
    else:
        raise ValueError(f"Unknown perturbation: {perturbation}")

    def apply(pil_img):
        arr = np.array(pil_img.convert("RGB"))
        return _PILImage.fromarray(aug(image=arr)["image"].astype(np.uint8))

    return apply


def _annotate_perturbation(img_bgr, perturb_type, severity, delta):
    """Overlay a dark banner in the top-left with perturbation info and delta_mAP50."""
    out = img_bgr.copy()
    h, w = out.shape[:2]
    bh = max(30, int(h * 0.052))
    overlay = out.copy()
    cv2.rectangle(overlay, (0, 0), (w, bh), (20, 20, 20), -1)
    out = cv2.addWeighted(overlay, 0.72, out, 0.28, 0)
    delta_str = f"{delta:+.4f}" if delta == delta else "N/A"  # NaN guard
    label = f"{perturb_type} [{severity}]   delta_mAP50={delta_str}"
    cv2.putText(
        out,
        label,
        (8, bh - 8),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )
    return out


# ── Pass E main loop ──────────────────────────────────────────────────────────
n_rendered = 0
n_skipped = 0

for ds, frames in manifest.items():
    for perturb_type in _PERTURB_TYPES:
        for severity in _SEVERITIES:
            condition = f"{perturb_type}_{severity}"
            out_dir = os.path.join(FRAMES_DIR, ds, "perturbations", condition)
            os.makedirs(out_dir, exist_ok=True)

            transform = _make_perturb_transform(perturb_type, severity)
            delta = _delta_lookup.get((ds, perturb_type, severity), float("nan"))

            for fkey, entry in frames.items():
                out_path = os.path.join(out_dir, f"{fkey}.png")
                if os.path.exists(out_path):
                    n_skipped += 1
                    continue

                img_path = entry["path"]
                if not os.path.exists(img_path):
                    print(f"  [WARN] missing source: {img_path}")
                    continue

                pil_img = _PILImage.open(img_path).convert("RGB")
                pil_perturb = transform(pil_img)

                # Resize for inference if image exceeds MAX_INFER_SIZE (e.g. mmCows 4480×2800)
                infer_img, _scale = cap_pil_for_inference(pil_perturb)
                dets = detector.predict(infer_img, threshold=CONF_THRESH)

                if dets and dets.xyxy is not None and len(dets.xyxy):
                    boxes = [[float(v) / _scale for v in row] for row in dets.xyxy]
                    scores = [float(s) for s in dets.confidence]
                else:
                    boxes, scores = [], []

                img_bgr = cv2.cvtColor(np.array(pil_perturb), cv2.COLOR_RGB2BGR)
                canvas = draw_boxes(img_bgr, boxes, scores)
                canvas = _annotate_perturbation(canvas, perturb_type, severity, delta)
                cv2.imwrite(out_path, canvas)
                n_rendered += 1
                print(
                    f"  [E] {ds}/{condition}/{fkey}: {len(boxes)} det, delta={delta:+.4f} → {out_path}"
                )

print(f"\nPass E complete — {n_rendered} rendered, {n_skipped} skipped.")

Loaded 60 delta entries from /home/sakif/TXST/Thesis/cattle-vision-framework/results/generalization/perturbation_delta.csv


  [E] mmcows/brightness_low/frame_001: 1 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/brightness_low/frame_001.png


  [E] mmcows/brightness_low/frame_002: 1 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/brightness_low/frame_002.png


  [E] mmcows/brightness_low/frame_003: 0 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/brightness_low/frame_003.png


  [E] mmcows/brightness_low/frame_004: 0 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/brightness_low/frame_004.png


...

  [E] mmcows/rain_low/frame_004: 2 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/rain_low/frame_004.png


  [E] mmcows/rain_high/frame_001: 4 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/rain_high/frame_001.png


  [E] mmcows/rain_high/frame_002: 4 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/rain_high/frame_002.png


  [E] mmcows/rain_high/frame_003: 3 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/rain_high/frame_003.png


  [E] mmcows/rain_high/frame_004: 2 det, delta=+nan → /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/mmcows/perturbations/rain_high/frame_004.png

Pass E complete — 40 rendered, 240 skipped.


## Manifest Update — Render Paths

Write render type → file path entries for every frame into `manifest.json`.


In [12]:
# Re-load and update manifest with render file paths
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

RENDER_SUBDIRS = {
    "detection": "detection",
    "det_seg": "det_seg",
    "det_seg_track": "det_seg_track",
    "full_pipeline": "full_pipeline",
}

for ds, frames in manifest.items():
    for fkey in frames:
        for key, subdir in RENDER_SUBDIRS.items():
            if key == "full_pipeline" and ds not in TRACKING_DATASETS:
                manifest[ds][fkey][key] = None
                continue
            p = os.path.join(FRAMES_DIR, ds, subdir, f"{fkey}.png")
            manifest[ds][fkey][key] = p if os.path.exists(p) else None

        # Perturbation render paths: perturbations → {condition: path_or_null}
        perturb_paths = {}
        for perturb_type in _PERTURB_TYPES:
            for severity in _SEVERITIES:
                condition = f"{perturb_type}_{severity}"
                p = os.path.join(
                    FRAMES_DIR, ds, "perturbations", condition, f"{fkey}.png"
                )
                perturb_paths[condition] = p if os.path.exists(p) else None
        manifest[ds][fkey]["perturbations"] = perturb_paths

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest updated: {MANIFEST_PATH}")
for ds, frames in manifest.items():
    for fkey, entry in frames.items():
        present = [k for k in RENDER_SUBDIRS if entry.get(k)]
        n_perturb = sum(1 for v in entry.get("perturbations", {}).values() if v)
        print(f"  {ds}/{fkey}: passes={present}, perturbations={n_perturb}/10")

Manifest updated: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/manifest.json
  cbvd5/frame_001: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cbvd5/frame_002: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cbvd5/frame_003: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cbvd5/frame_004: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cvb/frame_001: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cvb/frame_002: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cvb/frame_003: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  cvb/frame_004: passes=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'], perturbations=10/10
  opencows2020/frame_001: passes=['detection', 'det_seg',

In [1]:
import os, json, glob
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import FancyBboxPatch
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
FRAMES_DIR = os.path.join(REPO_ROOT, "results", "figures", "frames")
PLATES_DIR = os.path.join(REPO_ROOT, "results", "figures", "plates")
MANIFEST_PATH = os.path.join(FRAMES_DIR, "manifest.json")
TUBELETS_DIR = os.path.join(REPO_ROOT, "data", "processed", "tubelets")
LABELS_CSV = os.path.join(TUBELETS_DIR, "labels.csv")

os.makedirs(PLATES_DIR, exist_ok=True)

DATASET_ORDER = ["cbvd5", "cvb", "opencows2020", "cows2021", "cattleeyeview", "freeman", "mmcows"]
DATASET_LABELS = {
    "cbvd5": "CBVD-5",
    "cvb": "CVB",
    "opencows2020": "OpenCows2020",
    "cows2021": "Cows2021",
    "cattleeyeview": "CattleEyeView",
    "freeman": "Freeman",
    "mmcows": "mmCows",
}
RENDER_KEYS = ["detection", "det_seg", "det_seg_track", "full_pipeline"]
RENDER_LABELS = {
    "detection": "Detection",
    "det_seg": "Det + Seg",
    "det_seg_track": "Det + Seg + Track",
    "full_pipeline": "Full Pipeline",
}
LABEL_NAMES = {
    0: "Standing",
    1: "Lying",
    2: "Foraging",
    3: "Drinking",
    4: "Ruminating",
    5: "Grooming",
    6: "Other",
}
TRACKING_DATASETS = {"cbvd5", "cvb"}
DPI = 300


def load_img_rgb(path):
    """Load image as (H,W,3) float32 in [0,1]; return gray placeholder if missing."""
    if path and os.path.exists(path):
        img = mpimg.imread(path)
        if img.dtype == np.uint8:
            img = img.astype(np.float32) / 255.0
        else:
            img = img.astype(np.float32)
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        return img[:, :, :3]
    placeholder = np.full((180, 320, 3), 0.82, dtype=np.float32)
    return placeholder


def write_caption(plate_path, caption_text, label_slug):
    tex_path = plate_path.replace(".png", "_caption.tex")
    with open(tex_path, "w") as f:
        f.write(f"\\caption{{{caption_text}}}\\label{{fig:{label_slug}}}\n")
    return tex_path


print("REPO_ROOT:", REPO_ROOT)
print("PLATES_DIR:", PLATES_DIR)
print("Setup complete.")

REPO_ROOT: /home/sakif/TXST/Thesis/cattle-vision-framework
PLATES_DIR: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates
Setup complete.


## Quick-Verify

Read `manifest.json`, print per-dataset frame counts and which render types are present.  
Fails loudly if the manifest is missing.


In [2]:
if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}\n"
        "Run render_figures.ipynb (all cells) before this notebook."
    )

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print(f"Manifest: {MANIFEST_PATH}")
print(f"Datasets: {len(manifest)}\n")

total_renders = 0
for ds in DATASET_ORDER:
    frames = manifest.get(ds, {})
    if not frames:
        print(f"  {ds:16s}: MISSING from manifest")
        continue
    sample = list(frames.values())[0]
    present = [k for k in RENDER_KEYS if sample.get(k) and os.path.exists(sample[k])]
    missing = [
        k
        for k in RENDER_KEYS
        if not (sample.get(k) and os.path.exists(sample.get(k, "")))
    ]
    n_renders = len(present)
    total_renders += n_renders
    print(f"  {ds:16s}: {len(frames)} frames | present={present} | missing={missing}")

print(f"\nTotal render types present across all datasets: {total_renders}")
print("Note: missing renders will appear as gray placeholders in plates.")

Manifest: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/frames/manifest.json
Datasets: 7

  cbvd5           : 4 frames | present=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'] | missing=[]
  cvb             : 4 frames | present=['detection', 'det_seg', 'det_seg_track', 'full_pipeline'] | missing=[]
  opencows2020    : 4 frames | present=['detection', 'det_seg', 'det_seg_track'] | missing=['full_pipeline']
  cows2021        : 4 frames | present=['detection', 'det_seg', 'det_seg_track'] | missing=['full_pipeline']
  cattleeyeview   : 4 frames | present=['detection', 'det_seg', 'det_seg_track'] | missing=['full_pipeline']
  freeman         : 4 frames | present=['detection', 'det_seg', 'det_seg_track'] | missing=['full_pipeline']
  mmcows          : 4 frames | present=['detection', 'det_seg', 'det_seg_track'] | missing=['full_pipeline']

Total render types present across all datasets: 23
Note: missing renders will appear as gray placeholders in plates.


## Plate 1 — Pipeline Overview (6 × 4 grid)

Rows = 6 datasets, columns = 4 render types.  
One representative frame per cell (first frame from manifest).  
OOD datasets × full_pipeline show a gray N/A placeholder.  
Saved to `results/figures/plates/pipeline_overview_plate.png` at 300 DPI.


In [3]:
N_DS = len(DATASET_ORDER)
N_RENDER = len(RENDER_KEYS)
CELL_W, CELL_H = 3.4, 2.5

fig, axes = plt.subplots(
    N_DS,
    N_RENDER,
    figsize=(CELL_W * N_RENDER + 1.4, CELL_H * N_DS + 0.7),
)
fig.suptitle(
    "Pipeline Overview — All Datasets × All Render Stages",
    fontsize=11,
    fontweight="bold",
    y=1.002,
)

for row, ds in enumerate(DATASET_ORDER):
    frames = manifest.get(ds, {})
    rep_fkey = list(frames.keys())[0] if frames else None
    entry = frames[rep_fkey] if rep_fkey else {}

    for col, rkey in enumerate(RENDER_KEYS):
        ax = axes[row][col]
        path = entry.get(rkey)
        img = load_img_rgb(path)
        ax.imshow(img, aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_edgecolor("#ccc")

        if not (path and os.path.exists(path)):
            ax.text(
                0.5,
                0.5,
                "N/A",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=10,
                color="#888",
                style="italic",
            )

        if row == 0:
            ax.set_title(RENDER_LABELS[rkey], fontsize=8.5, pad=5)
        if col == 0:
            ax.set_ylabel(DATASET_LABELS[ds], fontsize=8.5, labelpad=6)

plt.tight_layout(rect=[0, 0, 1, 1.0])

out_path = os.path.join(PLATES_DIR, "pipeline_overview_plate.png")
plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

tex_path = write_caption(
    out_path,
    "Detection, segmentation, tracking, and full-pipeline overlay renders "
    "for one representative frame from each of the six evaluation datasets.",
    "pipeline-overview",
)
print(f"Caption: {tex_path}")

Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/pipeline_overview_plate.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/pipeline_overview_plate_caption.tex


/tmp/claude-1000/ipykernel_954724/2432412229.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Plate 2 — Pipeline Progression Strips (CBVD-5 and CVB)

For each of CBVD-5 and CVB: a 1×4 horizontal strip showing the same frame rendered
through each pipeline stage: detection → det+seg → det+seg+track → full pipeline.  
Saved as `pipeline_progression_{dataset}.png`.


In [4]:
for ds in ["cbvd5", "cvb"]:
    frames = manifest.get(ds, {})
    rep_fkey = list(frames.keys())[0] if frames else None
    entry = frames[rep_fkey] if rep_fkey else {}

    fig, axes = plt.subplots(1, N_RENDER, figsize=(N_RENDER * 4.5, 3.6))
    fig.suptitle(
        f'Pipeline Progression — {DATASET_LABELS[ds]} (frame {rep_fkey or "N/A"})',
        fontsize=10,
        fontweight="bold",
        y=1.02,
    )

    for col, rkey in enumerate(RENDER_KEYS):
        ax = axes[col]
        path = entry.get(rkey)
        img = load_img_rgb(path)
        ax.imshow(img, aspect="auto")
        ax.set_title(RENDER_LABELS[rkey], fontsize=9, pad=4)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_edgecolor("#ccc")

        if not (path and os.path.exists(path)):
            ax.text(
                0.5,
                0.5,
                "N/A",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=10,
                color="#888",
                style="italic",
            )

    plt.tight_layout()

    out_path = os.path.join(PLATES_DIR, f"pipeline_progression_{ds}.png")
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    tex_path = write_caption(
        out_path,
        f"Incremental pipeline stages applied to a representative "
        f"{DATASET_LABELS[ds]} frame: detection, segmentation, tracking, "
        "and full pipeline with behavior label.",
        f"pipeline-progression-{ds}",
    )
    print(f"Caption: {tex_path}")

/tmp/claude-1000/ipykernel_954724/3078125748.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/pipeline_progression_cbvd5.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/pipeline_progression_cbvd5_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/pipeline_progression_cvb.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/pipeline_progression_cvb_caption.tex


## Plate 3 — Behavior Class Examples (7 × 3 grid)

7 behavior classes (rows) × 3 tubelet frame examples (columns).  
The middle frame (frame_08.jpg) of each tubelet is used as the representative frame.  
Tubelets are sampled evenly across the labels.csv pool; val/test splits are preferred.  
Saved as `behavior_class_examples.png`.


In [5]:
N_EXAMPLES = 3
N_CLASSES = 7

df_labels = pd.read_csv(LABELS_CSV)
# Prefer val/test rows; fall back to any row if needed
df_val = df_labels[df_labels["split"].isin(["val", "test", "validation"])]
if len(df_val) == 0:
    df_val = df_labels


def pick_tubelet_frames(label_id, n=N_EXAMPLES):
    """Return n evenly-sampled middle-frame paths for the given label_id."""
    pool = df_val[df_val["label_id"] == label_id]
    if len(pool) == 0:
        pool = df_labels[df_labels["label_id"] == label_id]
    if len(pool) == 0:
        return [None] * n

    pool = pool.reset_index(drop=True)
    indices = np.linspace(0, len(pool) - 1, n, dtype=int)
    rows = [pool.iloc[i] for i in indices]

    paths = []
    for row in rows:
        tdir = os.path.join(REPO_ROOT, row["tubelet_dir"])
        # Prefer middle frame; fall back to first available
        mid = os.path.join(tdir, "frame_08.jpg")
        if os.path.exists(mid):
            paths.append(mid)
        else:
            frames = sorted(glob.glob(os.path.join(tdir, "frame_*.jpg")))
            paths.append(frames[len(frames) // 2] if frames else None)
    return paths


CELL_W3, CELL_H3 = 2.8, 2.5
fig, axes = plt.subplots(
    N_CLASSES,
    N_EXAMPLES,
    figsize=(CELL_W3 * N_EXAMPLES + 1.4, CELL_H3 * N_CLASSES + 0.5),
)
fig.suptitle(
    "Behavior Class Examples — Tubelet Middle Frames",
    fontsize=11,
    fontweight="bold",
    y=1.002,
)

for row, label_id in enumerate(range(N_CLASSES)):
    behavior = LABEL_NAMES[label_id]
    frame_paths = pick_tubelet_frames(label_id)

    for col, fpath in enumerate(frame_paths):
        ax = axes[row][col]
        img = load_img_rgb(fpath)
        ax.imshow(img, aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_edgecolor("#ccc")

        if not (fpath and os.path.exists(fpath)):
            ax.text(
                0.5,
                0.5,
                "N/A",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=9,
                color="#888",
                style="italic",
            )

        if col == 0:
            ax.set_ylabel(f"{label_id}: {behavior}", fontsize=8.5, labelpad=6)
        if row == 0:
            ax.set_title(f"Example {col + 1}", fontsize=8.5, pad=4)

plt.tight_layout(rect=[0, 0, 1, 1.0])

out_path = os.path.join(PLATES_DIR, "behavior_class_examples.png")
plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

tex_path = write_caption(
    out_path,
    "Three representative tubelet frames for each of the seven dairy cattle "
    "behavior classes used in VideoMAE training and evaluation.",
    "behavior-class-examples",
)
print(f"Caption: {tex_path}")

Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/behavior_class_examples.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/behavior_class_examples_caption.tex


/tmp/claude-1000/ipykernel_954724/3111893051.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Plate 4 — Perturbation Comparison Plates (one per perturbation type)

For each of the 5 perturbation types, a 6×2 grid: rows = 6 datasets, columns = low / high severity.
Each cell shows the detection overlay frame for that condition (delta*mAP50 already baked in by Pass E).
Saved as `perturbation_plate*{type}.png` at 300 DPI (5 plates total).


In [6]:
PERTURB_TYPES = ["brightness", "gaussian_noise", "motion_blur", "fog", "rain"]
SEVERITIES = ["low", "high"]
SEVERITY_LABELS = {"low": "Low Severity", "high": "High Severity"}
PERTURB_DISPLAY = {
    "brightness": "Brightness",
    "gaussian_noise": "Gaussian Noise",
    "motion_blur": "Motion Blur",
    "fog": "Fog",
    "rain": "Rain",
}

# Load delta_mAP50 for cell subtitles (values are already baked into frames,
# but we also print them as axes subtitles for clarity)
_delta_df = pd.read_csv(
    os.path.join(REPO_ROOT, "results", "generalization", "perturbation_delta.csv")
)
_delta_lookup = {
    (row.dataset, row.perturbation_type, row.severity): row.delta_mAP50
    for _, row in _delta_df.iterrows()
}


def perturb_frame_path(ds, perturb_type, severity, fkey="frame_001"):
    """Return path to a perturbation render, or None if not rendered yet."""
    p = os.path.join(
        FRAMES_DIR, ds, "perturbations", f"{perturb_type}_{severity}", f"{fkey}.png"
    )
    return p if os.path.exists(p) else None


for perturb_type in PERTURB_TYPES:
    N_SEV = len(SEVERITIES)
    CELL_W4, CELL_H4 = 3.6, 2.6

    fig, axes = plt.subplots(
        N_DS,
        N_SEV,
        figsize=(CELL_W4 * N_SEV + 1.4, CELL_H4 * N_DS + 0.7),
    )
    fig.suptitle(
        f"Perturbation — {PERTURB_DISPLAY[perturb_type]}: Low vs High Severity",
        fontsize=11,
        fontweight="bold",
        y=1.002,
    )

    for row, ds in enumerate(DATASET_ORDER):
        frames = manifest.get(ds, {})
        rep_fkey = list(frames.keys())[0] if frames else "frame_001"

        for col, severity in enumerate(SEVERITIES):
            ax = axes[row][col]
            path = perturb_frame_path(ds, perturb_type, severity, rep_fkey)
            img = load_img_rgb(path)
            delta = _delta_lookup.get((ds, perturb_type, severity), float("nan"))

            ax.imshow(img, aspect="auto")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(0.4)
                spine.set_edgecolor("#ccc")

            if not path:
                ax.text(
                    0.5,
                    0.5,
                    "N/A",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    fontsize=10,
                    color="#888",
                    style="italic",
                )

            # delta_mAP50 subtitle beneath each cell
            delta_str = (
                f"delta_mAP50={delta:+.4f}" if delta == delta else "delta_mAP50=N/A"
            )
            ax.set_xlabel(delta_str, fontsize=7.5, color="#444", labelpad=3)

            if row == 0:
                ax.set_title(SEVERITY_LABELS[severity], fontsize=8.5, pad=5)
            if col == 0:
                ax.set_ylabel(DATASET_LABELS[ds], fontsize=8.5, labelpad=6)

    plt.tight_layout(rect=[0, 0, 1, 1.0])

    out_path = os.path.join(PLATES_DIR, f"perturbation_plate_{perturb_type}.png")
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    tex_path = write_caption(
        out_path,
        f"Detection overlay renders under {PERTURB_DISPLAY[perturb_type].lower()} "
        "perturbation at low and high severity across all six evaluation datasets. "
        "Each frame is annotated with the delta\\textsubscript{mAP50} relative to the clean baseline.",
        f'perturbation-plate-{perturb_type.replace("_", "-")}',
    )
    print(f"Caption: {tex_path}")

print(f"\nPlate 4 complete — {len(PERTURB_TYPES)} plates written.")

/tmp/claude-1000/ipykernel_954724/2103730393.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_brightness.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_brightness_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_gaussian_noise.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_gaussian_noise_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_motion_blur.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_motion_blur_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_fog.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_fog_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_rain.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_plate_rain_caption.tex

Plate 4 complete — 5 plates written.


## Plate 5 — Freeman Center Severity Strip (1 × 11)

Leftmost cell: clean detection overlay for Freeman Center.  
Followed by all 10 perturbation conditions in order (brightness_low, brightness_high, …, rain_high).  
Caption below each cell shows condition name and delta_mAP50.  
Saved as `perturbation_severity_strip_freeman.png` at 300 DPI.


In [7]:
DS_STRIP = "freeman"
frames_strip = manifest.get(DS_STRIP, {})
rep_fkey_strip = list(frames_strip.keys())[0] if frames_strip else "frame_001"
entry_strip = frames_strip.get(rep_fkey_strip, {})

# Build ordered list of (label, path, delta) — clean first, then 10 conditions
CONDITIONS = [(p, s) for p in PERTURB_TYPES for s in SEVERITIES]

strip_cells = []  # (col_label, img_path, delta_str)

# Clean cell
clean_path = entry_strip.get("detection")
strip_cells.append(("clean\n(no perturbation)", clean_path, ""))

for perturb_type, severity in CONDITIONS:
    path = perturb_frame_path(DS_STRIP, perturb_type, severity, rep_fkey_strip)
    delta = _delta_lookup.get((DS_STRIP, perturb_type, severity), float("nan"))
    delta_str = f"{delta:+.4f}" if delta == delta else "N/A"
    label = f"{perturb_type}\n[{severity}]"
    strip_cells.append((label, path, delta_str))

N_STRIP = len(strip_cells)  # 11
CELL_W5, CELL_H5 = 2.8, 2.4

fig, axes = plt.subplots(1, N_STRIP, figsize=(CELL_W5 * N_STRIP, CELL_H5 + 1.0))
fig.suptitle(
    f"Freeman Center — Clean + All 10 Perturbation Conditions",
    fontsize=10,
    fontweight="bold",
    y=1.04,
)

for col, (label, path, delta_str) in enumerate(strip_cells):
    ax = axes[col]
    img = load_img_rgb(path)
    ax.imshow(img, aspect="auto")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
        spine.set_edgecolor("#ccc")

    if not (path and os.path.exists(path or "")):
        ax.text(
            0.5,
            0.5,
            "N/A",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=9,
            color="#888",
            style="italic",
        )

    title_text = label
    ax.set_title(title_text, fontsize=7.0, pad=3, linespacing=1.3)

    if delta_str:
        ax.set_xlabel(f"Δ={delta_str}", fontsize=7.0, color="#444", labelpad=3)

plt.tight_layout()

out_path = os.path.join(PLATES_DIR, "perturbation_severity_strip_freeman.png")
plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

tex_path = write_caption(
    out_path,
    "Freeman Center detection overlays under clean conditions and all ten perturbation "
    "conditions (five types $\\times$ two severity levels). "
    "$\\Delta$ values show delta\\textsubscript{mAP50} relative to the clean baseline.",
    "perturbation-strip-freeman",
)
print(f"Caption: {tex_path}")
print("\nPlate 5 complete.")

Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_severity_strip_freeman.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/perturbation_severity_strip_freeman_caption.tex

Plate 5 complete.


/tmp/claude-1000/ipykernel_954724/3952680249.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Plate 6 — Detection Gallery Strips (one per dataset)

For each of the 6 datasets, a 1×N horizontal strip (N = frames in manifest, typically 4)
showing all selected detection-only overlay renders.  
Each cell is labelled with its frame key.  
Saved as `detection_gallery_{dataset}.png` at 300 DPI (6 plates total).


In [8]:
CELL_W6, CELL_H6 = 3.4, 2.8

for ds in DATASET_ORDER:
    frames = manifest.get(ds, {})
    fkeys = list(frames.keys())
    N = len(fkeys)
    if N == 0:
        print(f"[WARN] No frames in manifest for {ds} — skipping")
        continue

    fig, axes = plt.subplots(1, N, figsize=(CELL_W6 * N, CELL_H6 + 0.7))
    if N == 1:
        axes = [axes]
    fig.suptitle(
        f"Detection Gallery — {DATASET_LABELS[ds]}",
        fontsize=10,
        fontweight="bold",
        y=1.02,
    )

    for col, fkey in enumerate(fkeys):
        ax = axes[col]
        path = os.path.join(FRAMES_DIR, ds, "detection", f"{fkey}.png")
        img = load_img_rgb(path)
        ax.imshow(img, aspect="auto")
        ax.set_title(fkey, fontsize=8.5, pad=4)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_edgecolor("#ccc")
        if not os.path.exists(path):
            ax.text(
                0.5,
                0.5,
                "N/A",
                transform=ax.transAxes,
                ha="center",
                va="center",
                fontsize=10,
                color="#888",
                style="italic",
            )

    plt.tight_layout()

    out_path = os.path.join(PLATES_DIR, f"detection_gallery_{ds}.png")
    plt.savefig(out_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    tex_path = write_caption(
        out_path,
        f"Detection-only overlay renders for all {N} selected "
        f"{DATASET_LABELS[ds]} frames. "
        "Bounding boxes and confidence scores are drawn by RF-DETR (threshold~0.3).",
        f'detection-gallery-{ds.replace("_", "-")}',
    )
    print(f"Caption: {tex_path}")

print(f"\nPlate 6 complete — {len(DATASET_ORDER)} detection gallery plates written.")

/tmp/claude-1000/ipykernel_954724/4009862542.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cbvd5.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cbvd5_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cvb.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cvb_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_opencows2020.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_opencows2020_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cows2021.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cows2021_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cattleeyeview.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_cattleeyeview_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_freeman.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_freeman_caption.tex


Saved: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_mmcows.png
Caption: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates/detection_gallery_mmcows_caption.tex

Plate 6 complete — 7 detection gallery plates written.


## Summary — All Plates Written


In [9]:
plate_files = sorted(glob.glob(os.path.join(PLATES_DIR, "*.png")))

print(f"Plates written to: {PLATES_DIR}")
print(f"{'File':<52} {'Size (KB)':>10} {'Pixels':>18}")
print("-" * 83)
for pf in plate_files:
    kb = os.path.getsize(pf) / 1024
    img = mpimg.imread(pf)
    h, w = img.shape[:2]
    name = os.path.basename(pf)
    print(f"  {name:<50} {kb:>9.0f} KB   {w:>5} × {h:<5}")

# Verify all expected plates are present
expected = [
    "pipeline_overview_plate.png",
    "pipeline_progression_cbvd5.png",
    "pipeline_progression_cvb.png",
    "behavior_class_examples.png",
    *[
        f"perturbation_plate_{t}.png"
        for t in ["brightness", "gaussian_noise", "motion_blur", "fog", "rain"]
    ],
    "perturbation_severity_strip_freeman.png",
    *[f"detection_gallery_{ds}.png" for ds in DATASET_ORDER],
]
written = {os.path.basename(p) for p in plate_files}
failures = [e for e in expected if e not in written]
print()
if failures:
    raise RuntimeError(f"Missing expected plates: {failures}")
else:
    print(f"All {len(expected)} expected plates present. ✓")

Plates written to: /home/sakif/TXST/Thesis/cattle-vision-framework/results/figures/plates
File                                                  Size (KB)             Pixels
-----------------------------------------------------------------------------------


  behavior_class_examples.png                             4078 KB    2904 × 5425 
  detection_gallery_cattleeyeview.png                     5339 KB    4049 × 1086 


  detection_gallery_cbvd5.png                             5765 KB    4049 × 1086 
  detection_gallery_cows2021.png                          4716 KB    4049 × 1086 
  detection_gallery_cvb.png                               3108 KB    4049 × 1086 


  detection_gallery_freeman.png                           4810 KB    4049 × 1086 
  detection_gallery_mmcows.png                            3021 KB    4049 × 1086 
  detection_gallery_opencows2020.png                      3959 KB    4049 × 1086 


  perturbation_plate_brightness.png                       5188 KB    2545 × 5693 


...

  perturbation_plate_motion_blur.png                     14805 KB    2545 × 5693 


  perturbation_plate_rain.png                            10510 KB    2545 × 5693 


  perturbation_severity_strip_freeman.png                 9216 KB    9210 × 1073 


  pipeline_overview_plate.png                             9534 KB    4465 × 5485 
  pipeline_progression_cbvd5.png                          2409 KB    5370 × 1116 


  pipeline_progression_cvb.png                            1530 KB    5370 × 1116 

All 17 expected plates present. ✓
